In [94]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.functions import when
from pyspark.sql.functions import col, when, sum as _sum

spark = SparkSession.builder.appName("Spotify Silver ETL").getOrCreate()

In [95]:
bronze_path = "../delta_lake/bronze/"

dim_artists = spark.read.parquet(bronze_path + "dim_artists")
dim_albums  = spark.read.parquet(bronze_path + "dim_albums")
dim_genres  = spark.read.parquet(bronze_path + "dim_genres")
dim_tracks  = spark.read.parquet(bronze_path + "dim_tracks")
fact_tracks = spark.read.parquet(bronze_path + "fact_tracks")

## Cleaning fact

In [96]:
fact_tracks.printSchema()

root
 |-- track_id: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- duration_ms: string (nullable = true)
 |-- explicit: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- tempo: string (nullable = true)
 |-- speechiness: string (nullable = true)
 |-- acousticness: string (nullable = true)
 |-- instrumentalness: string (nullable = true)
 |-- liveness: string (nullable = true)
 |-- valence: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- key: string (nullable = true)
 |-- time_signature: string (nullable = true)
 |-- ingest_date: timestamp (nullable = true)



In [97]:
fact_tracks = fact_tracks \
    .withColumn("popularity", col("popularity").cast("int")) \
    .withColumn("duration_ms", col("duration_ms").cast("int")) \
    .withColumn(
        "explicit",
        when(col("explicit") == "True", 1)
        .when(col("explicit") == "False", 0)
        .otherwise(None)
    ) \
    .withColumn("danceability", col("danceability").cast("double")) \
    .withColumn("energy", col("energy").cast("double")) \
    .withColumn("loudness", col("loudness").cast("double")) \
    .withColumn("tempo", col("tempo").cast("double")) \
    .withColumn("speechiness", col("speechiness").cast("double")) \
    .withColumn("acousticness", col("acousticness").cast("double")) \
    .withColumn("instrumentalness", col("instrumentalness").cast("double")) \
    .withColumn("liveness", col("liveness").cast("double")) \
    .withColumn("valence", col("valence").cast("double")) \
    .withColumn("mode", col("mode").cast("int")) \
    .withColumn("key", col("key").cast("int")) \
    .withColumn("time_signature", col("time_signature").cast("int"))

In [98]:
fact_tracks.show(5, truncate=False)
fact_tracks.printSchema()

+----------------------+----------+-----------+--------+------------+------+--------+-------+-----------+------------+----------------+--------+-------+----+---+--------------+--------------------------+
|track_id              |popularity|duration_ms|explicit|danceability|energy|loudness|tempo  |speechiness|acousticness|instrumentalness|liveness|valence|mode|key|time_signature|ingest_date               |
+----------------------+----------+-----------+--------+------------+------+--------+-------+-----------+------------+----------------+--------+-------+----+---+--------------+--------------------------+
|66ukyUZ0ygPwha3SIl7LkH|44        |414120     |0       |0.528       |0.774 |-5.87   |99.985 |0.037      |0.00297     |2.03E-6         |0.158   |0.213  |1   |6  |4             |2025-07-17 04:48:20.472144|
|7frJTPBH4qaatdpJXJfjB5|44        |889066     |0       |0.447       |0.32  |-10.914 |124.428|0.0306     |0.813       |0.0             |0.176   |0.148  |1   |2  |4             |2025-07-

In [99]:
fact_tracks.select([
    _sum(col(c).isNull().cast("int")).alias(c)
    for c in fact_tracks.columns
]).show()

+--------+----------+-----------+--------+------------+------+--------+-----+-----------+------------+----------------+--------+-------+----+---+--------------+-----------+
|track_id|popularity|duration_ms|explicit|danceability|energy|loudness|tempo|speechiness|acousticness|instrumentalness|liveness|valence|mode|key|time_signature|ingest_date|
+--------+----------+-----------+--------+------------+------+--------+-----+-----------+------------+----------------+--------+-------+----+---+--------------+-----------+
|       0|         0|          0|       0|           0|     0|       0|    0|          0|           0|               0|       0|      0|   0|  0|             0|          0|
+--------+----------+-----------+--------+------------+------+--------+-----+-----------+------------+----------------+--------+-------+----+---+--------------+-----------+



In [100]:
fact_tracks.write.mode("overwrite").parquet("../delta_lake/silver/fact_tracks") 

## Cleaning dim

### dim_artist

In [101]:
dim_artists.printSchema()

root
 |-- artists: string (nullable = true)
 |-- artist_id: string (nullable = true)
 |-- ingest_date: timestamp (nullable = true)



In [102]:
dim_artists = dim_artists.withColumn("artist_id", col("artist_id").cast("int"))
dim_artists = dim_artists.dropna()

In [103]:
dim_artists.show(5, truncate=False)
dim_artists.printSchema()

+----------------------+---------+--------------------------+
|artists               |artist_id|ingest_date               |
+----------------------+---------+--------------------------+
|Gen Hoshino           |1        |2025-07-17 04:48:18.283555|
|Ben Woodward          |2        |2025-07-17 04:48:18.283555|
|Ingrid Michaelson;ZAYN|3        |2025-07-17 04:48:18.283555|
|Kina Grannis          |4        |2025-07-17 04:48:18.283555|
|Chord Overstreet      |5        |2025-07-17 04:48:18.283555|
+----------------------+---------+--------------------------+
only showing top 5 rows
root
 |-- artists: string (nullable = true)
 |-- artist_id: integer (nullable = true)
 |-- ingest_date: timestamp (nullable = true)



In [104]:
dim_artists.write.mode("overwrite").parquet("../delta_lake/silver/dim_artists") 

### dim_albums

In [105]:
dim_albums.printSchema()

root
 |-- album_id: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- artist_id: string (nullable = true)
 |-- ingest_date: timestamp (nullable = true)



In [106]:
dim_albums = dim_albums \
    .withColumn("album_id", col("album_id").cast("int")) \
    .withColumn("artist_id", col("artist_id").cast("int"))

dim_albums = dim_albums.dropna()

In [107]:
dim_albums.show(5, truncate=False)
dim_albums.printSchema()

+--------+------------------------------------------------------+---------+--------------------------+
|album_id|album_name                                            |artist_id|ingest_date               |
+--------+------------------------------------------------------+---------+--------------------------+
|1       |Comedy                                                |1        |2025-07-17 04:48:18.514609|
|2       |Ghost (Acoustic)                                      |2        |2025-07-17 04:48:18.514609|
|3       |To Begin Again                                        |3        |2025-07-17 04:48:18.514609|
|4       |Crazy Rich Asians (Original Motion Picture Soundtrack)|4        |2025-07-17 04:48:18.514609|
|5       |Hold On                                               |5        |2025-07-17 04:48:18.514609|
+--------+------------------------------------------------------+---------+--------------------------+
only showing top 5 rows
root
 |-- album_id: integer (nullable = true)
 |-

In [108]:
dim_albums.write.mode("overwrite").parquet("../delta_lake/silver/dim_albums") 

### dim_genres

In [109]:
dim_genres.printSchema()

root
 |-- track_genre: string (nullable = true)
 |-- genre_id: string (nullable = true)
 |-- ingest_date: timestamp (nullable = true)



In [110]:
dim_genres = dim_genres.withColumn("genre_id", col("genre_id").cast("int"))
dim_genres = dim_genres.dropna()

In [111]:
dim_genres.show(5, truncate=False)
dim_genres.printSchema()

+-----------+--------+-------------------------+
|track_genre|genre_id|ingest_date              |
+-----------+--------+-------------------------+
|acoustic   |1       |2025-07-17 04:48:18.82478|
|afrobeat   |2       |2025-07-17 04:48:18.82478|
|alt-rock   |3       |2025-07-17 04:48:18.82478|
|alternative|4       |2025-07-17 04:48:18.82478|
|ambient    |5       |2025-07-17 04:48:18.82478|
+-----------+--------+-------------------------+
only showing top 5 rows
root
 |-- track_genre: string (nullable = true)
 |-- genre_id: integer (nullable = true)
 |-- ingest_date: timestamp (nullable = true)



In [112]:
dim_genres.write.mode("overwrite").parquet("../delta_lake/silver/dim_genres") 

### dim_tracks

In [113]:
dim_tracks.printSchema()

root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- album_id: string (nullable = true)
 |-- genre_id: string (nullable = true)
 |-- ingest_date: timestamp (nullable = true)



In [115]:
dim_tracks = dim_tracks \
    .withColumn("album_id", col("album_id").cast("int")) \
    .withColumn("genre_id", col("genre_id").cast("int"))

dim_tracks = dim_tracks.dropna()

In [116]:
dim_tracks.show(5, truncate=False)
dim_tracks.printSchema()

+----------------------+--------------------------+--------+--------+--------------------------+
|track_id              |track_name                |album_id|genre_id|ingest_date               |
+----------------------+--------------------------+--------+--------+--------------------------+
|5SuOikwiRyPMVoIQDJUgSV|Comedy                    |1       |1       |2025-07-17 04:48:18.927599|
|4qPNDBW1i3p13qLCt0Ki3A|Ghost - Acoustic          |2       |1       |2025-07-17 04:48:18.927599|
|1iJBSr7s7jYXzM8EGcbK5b|To Begin Again            |3       |1       |2025-07-17 04:48:18.927599|
|6lfxq3CG4xtTiEg7opyCyx|Can't Help Falling In Love|4       |1       |2025-07-17 04:48:18.927599|
|5vjLSffimiIP26QG5WcN2K|Hold On                   |5       |1       |2025-07-17 04:48:18.927599|
+----------------------+--------------------------+--------+--------+--------------------------+
only showing top 5 rows
root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- album_id: i

In [117]:
dim_tracks.write.mode("overwrite").parquet("../delta_lake/silver/dim_tracks")